# Zero-shot classification of textual variants

**Goal:** sort each variant in our corpus into one of three categories (**error**, **substantive**, **vocabulary**) *without training a model ourselves*.

**Model:** [`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli), a ~400M-parameter language model fine-tuned for *Natural Language Inference* (NLI).

### How does "zero-shot" work?
The model was trained on one task only: given a **premise** and a **hypothesis**, is the hypothesis
*entailed*, *neutral*, or *contradicted*?

We reuse that skill for classification:

| | Example |
|---|---|
| **Premise** (our data) | *In the manuscript, "kign" was replaced with "king".* |
| **Hypotheses** (one per category, written with the two readings) | *"kign" is a misspelling of "king".* / *"king" means the same as "kign".* / *"king" means something different from "kign".* |

The model scores how strongly the premise **entails** each hypothesis. The best-scoring hypothesis gives the predicted category.
"Zero-shot" = the model has never seen our categories; we only describe them in words.

For more info, see [readme](readme.md).

## Load the corpus
The settings below match our CSV file:
- `DEL_COL`: the column with the deleted reading (set to `None` if you don't have one).
- `ADD_COL`: the column with the added reading.
- `GOLD_COL`: a column with the category set manually in your ground truth (used for evaluation here below; otherwise `None`).

In [23]:
import pandas as pd


CSV_PATH = "gt.csv"   # path to the file, can be changed to gt.csv for the real data or gt_fake.csv for the fake data
SEP      = ","              # use ";" or "\t" depending on how the file is separated
DEL_COL = "del_full"           # or None
ADD_COL = "add_full"
DEL_CONTEXT = "del_in_context"  # or None
ADD_CONTEXT = "add_in_context"  # or None
GOLD_COL = "category"         # or None

try:
    df = pd.read_csv(CSV_PATH, sep=SEP)
    print(f"Loaded {len(df)} variants from {CSV_PATH}")
except FileNotFoundError:
    print("File not found.")

df.head() # display the first few rows of the dataframe (table)

Loaded 31 variants from gt.csv


,mod_id,notebook,folio,line_n,add_hand,word_scope,del_full,add_full,del_in_context,add_in_context,hand,n_removed,n_added,letter_similarity,case_only,same_lemma,category,Notes
0,ox-ms_abinger_c57-0061-mod002,ox-ms_abinger_c57,31r,5,NaN,word,tore,tearing,fled â On seeing me he darted towards me and...,fled â On seeing me he darted towards me and...,not specified,1,1,0.29,0,1,error,NaN
1,ox-ms_abinger_c57-0061-mod004,ox-ms_abinger_c57,31r,22,NaN,word,softness,gentleness,the miserable pain of & The feelings of kindne...,the miserable pain of & The feelings of kindne...,not specified,1,1,0.50,0,0,vocabulary,NaN
2,ox-ms_abinger_c57-0061-mod005,ox-ms_abinger_c57,31r,27,NaN,word,pain,the agony of my wound,vowed eternal hatred & vengeance to all mankin...,vowed eternal hatred & vengeance to all mankin...,not specified,1,5,0.10,0,0,substantive,NaN
3,ox-ms_abinger_c57-0062-mod002,ox-ms_abinger_c57,31v,3,#pbs,word,prayers,vows,of the injustice & ingratitude of their inflic...,of the injustice & ingratitude of their inflic...,#pbs,1,1,0.14,0,0,vocabulary,NaN
4,ox-ms_abinger_c57-0062-mod003,ox-ms_abinger_c57,31v,6,#pbs,word,horrors of my situation.,the outrages & the anguish I had endured.,& deadly revenge such as would alone compensat...,& deadly revenge such as would alone compensat...,#pbs,4,7,0.26,0,0,substantive,NaN


## Build the premise
The model reads one sentence per variant: the **premise**.

The premise is part of the prompt, but it’s not necessarily the whole prompt. It’s the sentence or context the model is given; the prompt can also include instructions about what to do with it.

Two choices here:
- **Only the change**, without the surrounding sentence: for this small model, more text means more distraction.
- **A neutral verb**: *replaced*. A verb like *corrected* would already suggest that the deleted reading was an error.

The commented lines are earlier versions, kept for comparison.

In [24]:
def make_premise(row):
    # return f'The original reads "{row[DEL_COL]}" in context "{row[DEL_CONTEXT]}"; the variant reads "{row[ADD_COL]}" in context "{row[ADD_CONTEXT]}".'
    # return f'The original reads "{row[DEL_COL]}"; the variant reads "{row[ADD_COL]}".'
    # return f'"{row[DEL_COL]}" was deleted and "{row[ADD_COL]}" was added.'
    # return f'The difference between "{row[DEL_COL]}" and "{row[ADD_COL]}".'
    return f'In the manuscript, "{row[DEL_COL]}" was replaced with "{row[ADD_COL]}".'

df["premise"] = df.apply(make_premise, axis=1)
df["premise"].iloc[0]

'In the manuscript, "tore" was replaced with "tearing".'

## Write the hypotheses
Each hypothesis is a **claim about the two readings of this variant**, which the premise can support or not:

| Category | Hypothesis |
|---|---|
| error | *"igmmatic" is a mistake, corrected in "enigmatic".* |
| vocabulary | *"enigmatic" means the same as "igmmatic".* |
| substantive | *"enigmatic" means something different from "igmmatic".* |

This is much closer to the sentence pairs the model saw during its training.
Because the hypotheses change with every variant, we write a function that builds them.

In [25]:
def make_hypotheses(row):
    # Returns one hypothesis per category, written with the two readings of this variant.
    deleted, added = row[DEL_COL], row[ADD_COL]
    return {
        "error":       f'"{deleted}" is a mistake (word does not exist or is incorrect), corrected in "{added}".',
        "vocabulary":  f'"{added}" means something similar to "{deleted}". ',
        "substantive": f'"{added}" means something different from "{deleted}".',
    }

for category, hypothesis in make_hypotheses(df.iloc[0]).items():
    print(f"{category:<12} {hypothesis}")

error        "tore" is a mistake (word does not exist or is incorrect), corrected in "tearing".
vocabulary   "tearing" means something similar to "tore". 
substantive  "tearing" means something different from "tore".


## Load the model and test one example
The first run downloads the model (~1.6 GB). It runs on CPU; a GPU is faster but not required.

In [ ]:
from transformers import pipeline


classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def classify(row):
    # Returns the score of each category for one variant.
    hypotheses = make_hypotheses(row)
    result = classifier(
        row["premise"],
        list(hypotheses.values()),  # The hypotheses are the candidate "labels".
        hypothesis_template="{}",   # They are already full sentences: nothing to add around them.
        multi_label=False,           # The hypotheses compete: their scores sum to 1.
    )
    score_of = dict(zip(result["labels"], result["scores"]))  # hypothesis -> score
    return {category: round(score_of[h], 3) for category, h in hypotheses.items()}

# Test on two variants.
for i in [6, 20]:
    row = df.iloc[i]
    print("Premise:", row["premise"])
    for category, score in classify(row).items():
        print(f"  {category:<12} {score:.3f}")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4210.20it/s]


Premise: In the manuscript, "Geneva" was replaced with "that town".
  error        0.316
  vocabulary   0.368
  substantive  0.316
Premise: In the manuscript, "know" was replaced with "have learned".
  error        0.177
  vocabulary   0.506
  substantive  0.317


With `multi_label=True`, each hypothesis is scored on its own, from 0 to 1: "how likely is it that this is a misspelling?", "how likely is it that the two readings mean the same?", and so on. The scores do not sum to 1. We pick the highest.

With `multi_label=False`, the categories competed with each other and the scores summed to 1.

## Classify the whole corpus
For each variant we keep the **predicted category** and the **confidence** (score of the winner).

In [27]:
scores = pd.DataFrame([classify(row) for _, row in df.iterrows()], index=df.index)  # One row per variant, one column per category.

df["predicted"]  = scores.idxmax(axis=1)  # The category with the highest score.
df["confidence"] = scores.max(axis=1)     # Its score.
for category in scores.columns:
    df["score_" + category] = scores[category]  # Keeps all the scores, to look at them later.

df[[DEL_COL, ADD_COL, "score_error", "score_vocabulary", "score_substantive", "predicted"]]

,del_full,add_full,score_error,score_vocabulary,score_substantive,predicted
0,tore,tearing,0.207,0.611,0.182,vocabulary
1,softness,gentleness,0.087,0.641,0.272,vocabulary
2,pain,the agony of my wound,0.110,0.402,0.488,substantive
3,prayers,vows,0.097,0.528,0.375,vocabulary
4,horrors of my situation.,the outrages & the anguish I had endured.,0.167,0.307,0.527,substantive
5,journey,toils,0.159,0.474,0.367,vocabulary
6,Geneva,that town,0.316,0.368,0.316,vocabulary
7,gazen,gazed,0.213,0.493,0.294,vocabulary
8,athought,an idea,0.175,0.440,0.386,vocabulary
9,hav e,had,0.262,0.489,0.249,vocabulary


In [28]:
# How many variants per category?
df["predicted"].value_counts()

predicted
vocabulary     27
substantive     4
Name: count, dtype: int64

In [29]:
# Uncertain cases deserve a human look (threshold is arbitrary: adjust it)
df[df["confidence"] < 0.6][[c for c in [DEL_COL, ADD_COL, "predicted", "confidence"] if c]]

,del_full,add_full,predicted,confidence
2,pain,the agony of my wound,substantive,0.488
3,prayers,vows,vocabulary,0.528
4,horrors of my situation.,the outrages & the anguish I had endured.,substantive,0.527
5,journey,toils,vocabulary,0.474
6,Geneva,that town,vocabulary,0.368
7,gazen,gazed,vocabulary,0.493
8,athought,an idea,vocabulary,0.440
9,hav e,had,vocabulary,0.489
10,horrow,horror,vocabulary,0.531
11,on,in,vocabulary,0.545


## Evaluate (only if you have manual labels)
Compare the model with **your** judgement.
- **Accuracy**: share of variants where model and human agree.
- **Confusion matrix**: rows = human label, columns = model label. Off-diagonal cells show *which* categories the model confuses.

In [30]:
if GOLD_COL and GOLD_COL in df.columns:
    # Compare the model's predictions with the manual labels.
    # The accuracy is the percentage of predictions that match.
    accuracy = (df["predicted"] == df[GOLD_COL]).mean()
    print(f"Accuracy: {accuracy:.0%}")
    display(pd.crosstab(df[GOLD_COL], df["predicted"], rownames=["human"], colnames=["model"]))
else:
    print("No manual labels: skipping evaluation.")

Accuracy: 39%


model,substantive,vocabulary
human,,
error,0,12
substantive,2,4
vocabulary,1,10


## Save the results

In [31]:
df.drop(columns="premise").to_csv("variants_classified_zero_shot_all.csv", index=False)
print("Saved to variants_classified_zero_shot_all.csv")

Saved to variants_classified_zero_shot_all.csv


## Things to keep in mind
- **Language:** this model was trained on **English**. For texts in other languages, try a multilingual NLI model, e.g. `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (change only the `model=` name).
- **Confidence ≠ correctness:** a model can be confidently wrong.
- **Prompt sensitivity:** small changes in the premise or the hypotheses can change the results noticeably.
- **Spelling is hard for this model:** this is a transformer model, that cuts unknown words like "igmmatic" into pieces and has no real notion of spelling.

### Exercises
1. Rewrite the hypotheses, e.g. *"{deleted}" is a spelling error* or *"{added}" is a synonym of "{deleted}"*. Which wording works best?
2. Look at the misclassified variants. Is there a pattern?